# Benchmark Auswertung: Multiprocessing
Baseline vs. Static vs. Dynamic vs. Dynamic True

Wird ausgeführt sobald alle drei Laptops ihre Ergebnisse gepusht haben.

#### Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from pathlib import Path
from scipy import stats
import ast

PROCESS_COUNTS = [1, 2, 4, 8]
IMAGE_COUNTS   = [100, 500, 1000, 5000]

colors  = {"Baseline": "green", "Static": "steelblue", "Dynamic": "darkorange", "Dynamic True": "purple"}
markers = {"Baseline": "^",     "Static": "o",         "Dynamic": "s",           "Dynamic True": "D"}
lines   = {1: "-", 2: "--", 3: ":"}

Path("results").mkdir(exist_ok=True)

#### 1. Daten laden

In [ ]:
all_csvs = sorted(glob.glob("results/laptop*/*.csv"))
print(f"Gefundene CSVs: {len(all_csvs)}")
for f in all_csvs:
    print(f"  {f}")

df = pd.concat([pd.read_csv(f) for f in all_csvs], ignore_index=True)
print(f"\nZeilen gesamt: {len(df)}")
df.groupby(["Laptop", "Variante", "Datensatz"])["Bilder"].count()

#### 2. Laufzeit bei steigender Bildanzahl (alle Laptops)

In [ ]:
datasets = ["NIH", "Kaggle Pneumonia"]
laptops  = sorted(df["Laptop"].unique())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Laufzeit bei steigender Bildanzahl (8 Prozesse, alle Laptops)")

for ax, ds in zip(axes, datasets):
    for laptop in laptops:
        for variant in ["Baseline", "Static", "Dynamic", "Dynamic True"]:
            sub = df[
                (df["Laptop"]    == laptop) &
                (df["Variante"]  == variant) &
                (df["Datensatz"] == ds) &
                (df["Prozesse"]  == (1 if variant == "Baseline" else 8))
            ]
            if not sub.empty:
                ax.errorbar(
                    sub["Bilder"], sub["Laufzeit in s"],
                    yerr=sub["Std"] if "Std" in sub.columns else None,
                    color=colors[variant],
                    marker=markers[variant],
                    linestyle=lines.get(laptop, "-"),
                    capsize=4,
                    label=f"{variant} L{laptop}"
                )
    ax.set_title(ds)
    ax.set_xlabel("Bildanzahl")
    ax.set_ylabel("Laufzeit in s")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("results/runtime_by_image_count_all_laptops.png", dpi=150)
plt.show()

Baseline liegt auf allen Laptops deutlich über Static und Dynamic. Das zeigt den klaren Gewinn durch Parallelisierung. Bei NIH braucht die Baseline bei 5000 Bildern ~30-39s, Static und Dynamic kommen auf ~8-10s -> Faktor 4-5 schneller. Bei Kaggle Pneumonia ähnlich, aber hier trennen sich Static und Dynamic: Static ist langsamer als Dynamic, weil variable Auflösungen zu Lastungleichgewicht führen. L3 ist auf beiden Datensätzen der schnellste Laptop.

#### 3. Speedup — alle Laptops

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Speedup bei 2, 4 und 8 Prozessen (1000 Bilder, alle Laptops)")

for ax, ds in zip(axes, datasets):
    for laptop in laptops:
        for variant in ["Static", "Dynamic", "Dynamic True"]:
            sub = df[
                (df["Laptop"]    == laptop) &
                (df["Variante"]  == variant) &
                (df["Datensatz"] == ds) &
                (df["Bilder"]    == 1000)
            ]
            if not sub.empty:
                ax.errorbar(
                    sub["Prozesse"], sub["Speedup"],
                    yerr=sub["Std"] if "Std" in sub.columns else None,
                    color=colors[variant],
                    marker=markers[variant],
                    linestyle=lines.get(laptop, "-"),
                    capsize=4,
                    label=f"{variant} L{laptop}"
                )
    ax.plot(PROCESS_COUNTS, PROCESS_COUNTS, "k--", alpha=0.3, label="Ideal")
    ax.set_title(ds)
    ax.set_xlabel("Prozesse")
    ax.set_ylabel("Speedup")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("results/speedup_by_processes_all_laptops.png", dpi=150)
plt.show()

Kein Laptop erreicht linearen Speedup. Bei 8 Prozessen maximal ~6.8 statt idealem 8. L1 und L2 liegen bei ~3.3-4.2, L3 deutlich höher bei ~5.5-6.8. Static und Dynamic liegen auf beiden Datensätzen eng beieinander.

#### 4. Efficiency — alle Laptops

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Efficiency bei 2, 4 und 8 Prozessen (1000 Bilder, alle Laptops)")

for ax, ds in zip(axes, datasets):
    for laptop in laptops:
        for variant in ["Static", "Dynamic", "Dynamic True"]:
            sub = df[
                (df["Laptop"]    == laptop) &
                (df["Variante"]  == variant) &
                (df["Datensatz"] == ds) &
                (df["Bilder"]    == 1000)
            ]
            if not sub.empty:
                ax.errorbar(
                    sub["Prozesse"], sub["Efficiency"],
                    yerr=sub["Std"] if "Std" in sub.columns else None,
                    color=colors[variant],
                    marker=markers[variant],
                    linestyle=lines.get(laptop, "-"),
                    capsize=4,
                    label=f"{variant} L{laptop}"
                )
    ax.axhline(1.0, color="k", linestyle="--", alpha=0.3, label="Ideal")
    ax.set_title(ds)
    ax.set_xlabel("Prozesse")
    ax.set_ylabel("Efficiency")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("results/efficiency_by_processes_all_laptops.png", dpi=150)
plt.show()

Efficiency sinkt auf allen Laptops mit steigender Prozessanzahl. Bei 8 Prozessen liegt L1 und L2 bei ~41-53%, L3 hält ~84%. Dynamic ist tendenziell etwas effizienter als Static, besonders bei Kaggle Pneumonia wo der Lastausgleich greift.

#### 5. Static vs. Dynamic vs. Baseline — alle Laptops

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Static vs. Dynamic vs. Baseline — Laufzeit bei 5000 Bildern, alle Laptops")

for ax, ds in zip(axes, datasets):
    for laptop in laptops:
        sub_base = df[
            (df["Laptop"]    == laptop) &
            (df["Variante"]  == "Baseline") &
            (df["Datensatz"] == ds) &
            (df["Bilder"]    == 5000)
        ]
        if not sub_base.empty:
            ax.axhline(
                sub_base["Laufzeit in s"].values[0],
                color=colors["Baseline"],
                linestyle=lines.get(laptop, "-"),
                alpha=0.7,
                label=f"Baseline L{laptop}"
            )
        for variant in ["Static", "Dynamic", "Dynamic True"]:
            sub = df[
                (df["Laptop"]    == laptop) &
                (df["Variante"]  == variant) &
                (df["Datensatz"] == ds) &
                (df["Bilder"]    == 5000)
            ]
            if not sub.empty:
                ax.errorbar(
                    sub["Prozesse"], sub["Laufzeit in s"],
                    yerr=sub["Std"] if "Std" in sub.columns else None,
                    color=colors[variant],
                    marker=markers[variant],
                    linestyle=lines.get(laptop, "-"),
                    capsize=4,
                    label=f"{variant} L{laptop}"
                )
    ax.set_title(ds)
    ax.set_xlabel("Prozesse")
    ax.set_ylabel("Laufzeit in s")
    ax.legend(fontsize=7)
    ax.grid(True)

plt.tight_layout()
plt.savefig("results/static_vs_dynamic_all_laptops.png", dpi=150)
plt.show()

Bei NIH liegen Static und Dynamic eng beieinander, beide weit unter der Baseline. Bei Kaggle Pneumonia ist der Abstand zwischen Static und Dynamic klar sichtbar auf allen drei Laptops: Dynamic ist ~20-30% schneller als Static. Das bestätigt die Hypothese.

#### 6. Amdahls Gesetz

In [ ]:
def amdahl(p, f=0.95):
    return 1 / ((1 - f) + f / p)

p_range = np.linspace(1, 8, 100)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Amdahl vs. Empirisch (1000 Bilder, alle Laptops)")

for ax, ds in zip(axes, datasets):
    ax.plot(p_range, [amdahl(p) for p in p_range], "k--", alpha=0.5, label="Amdahl (f=0.95)")
    ax.plot(PROCESS_COUNTS, PROCESS_COUNTS, "gray", linestyle=":", alpha=0.3, label="Ideal (linear)")
    for laptop in laptops:
        for variant in ["Static", "Dynamic", "Dynamic True"]:
            sub = df[
                (df["Laptop"]    == laptop) &
                (df["Variante"]  == variant) &
                (df["Datensatz"] == ds) &
                (df["Bilder"]    == 1000)
            ]
            if not sub.empty:
                ax.errorbar(
                    sub["Prozesse"], sub["Speedup"],
                    yerr=sub["Std"] if "Std" in sub.columns else None,
                    color=colors[variant],
                    marker=markers[variant],
                    linestyle=lines.get(laptop, "-"),
                    capsize=4,
                    label=f"{variant} L{laptop}"
                )
    ax.set_title(ds)
    ax.set_xlabel("Prozesse")
    ax.set_ylabel("Speedup")
    ax.legend(fontsize=8)
    ax.grid(True)

plt.tight_layout()
plt.savefig("results/amdahl_all_laptops.png", dpi=150)
plt.show()

L1 und L2 liegen unterhalb der Amdahl-Kurve mit f=0.95. Der sequenzieller Anteil ist größer als 5%, also mehr Overhead als angenommen. L3 Dynamic übertrifft die Amdahl-Kurve, d.h. die Hardware skaliert besser als das Modell vorhersagt.

#### 7. Ergebnistabelle

In [ ]:
cols = ["Laptop", "Variante", "Datensatz", "Bilder", "Prozesse",
        "Laufzeit in s", "Speedup", "Efficiency", "Throughput"]

for col in ["Std", "CI_low", "CI_high"]:
    if col in df.columns:
        cols.append(col)

table = df[cols].sort_values(["Laptop", "Variante", "Datensatz", "Bilder", "Prozesse"])
print(table.to_markdown(index=False))

#### T Test

In [ ]:
results_ttest = []

for ds in ["NIH", "Kaggle Pneumonia"]:
    for n_img in IMAGE_COUNTS:
        for n_proc in PROCESS_COUNTS:

            row_static = df[
                (df["Variante"]  == "Static") &
                (df["Datensatz"] == ds) &
                (df["Bilder"]    == n_img) &
                (df["Prozesse"]  == n_proc)
            ]
            row_true = df[
                (df["Variante"]  == "Dynamic True") &
                (df["Datensatz"] == ds) &
                (df["Bilder"]    == n_img) &
                (df["Prozesse"]  == n_proc)
            ]

            if row_static.empty or row_true.empty:
                continue
            if "all_times" not in df.columns:
                print("all_times Spalte fehlt — Benchmark neu ausführen")
                break

            times_static = ast.literal_eval(row_static["all_times"].values[0])
            times_true   = ast.literal_eval(row_true["all_times"].values[0])

            t_stat, p_value = stats.ttest_ind(times_static, times_true)

            results_ttest.append({
                "Datensatz":           ds,
                "Bilder":              n_img,
                "Prozesse":            n_proc,
                "Static Median":       round(float(row_static["Laufzeit in s"].values[0]), 3),
                "Dynamic True Median": round(float(row_true["Laufzeit in s"].values[0]), 3),
                "p-value":             round(p_value, 4),
                "Signifikant":         "ja" if p_value < 0.05 else "nein",
            })

df_ttest = pd.DataFrame(results_ttest)
print(df_ttest.to_markdown(index=False))